In [0]:
# Load your existing CSV
temp_df = spark.read.format("csv").option("header", "true").load("/Volumes/workspace/raw_data/results/results.csv")

# Save it back as a JSON file in the same Volume (write to a directory, not a file)
temp_df.write.format("json") \
    .mode("overwrite") \
    .save("/Volumes/workspace/raw_data/results/results_json/")

In [0]:
# Point this to the folder you wrote JSON to
json_path = "/Volumes/workspace/raw_data/results/results_json/"

results_df = spark.read.format("json") \
    .option("inferSchema", "true") \
    .load(json_path)

# Check if the data loaded correctly
results_df.printSchema()
display(results_df.limit(10))

In [0]:
# Try reading your converted file again with this EXTRA option
results_df = spark.read.format("json") \
    .option("multiLine", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/raw_data/results/results.json")

display(results_df)

In [0]:
# 1. Verify the file exists in the new 'results' volume
display(dbutils.fs.ls("/Volumes/workspace/raw_data/results/"))

# 2. Read from the CORRECT location
csv_path = "/Volumes/workspace/raw_data/results/results.csv"

csv_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(csv_path)

# 3. Create the JSON folder (Transformation)
csv_df.write.format("json") \
    .mode("overwrite") \
    .save("/Volumes/workspace/raw_data/results/results_json_fix")

display(csv_df)

In [0]:
from pyspark.sql.functions import concat, col, lit, current_timestamp

# 1. Read the CSV file
drivers_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/raw_data/drivers/drivers.csv")

# 2. Rename columns
drivers_silver_df = drivers_df \
    .withColumnRenamed("driverId", "driver_id") \
    .withColumnRenamed("driverRef", "driver_ref") \
    .withColumn("name", concat(col("forename"), lit(" "), col("surname"))) \
    .withColumn("nationality", col("nationality")) \
    .withColumn("ingestion_date", current_timestamp()) \
    .select("driver_id", "driver_ref", "name", "nationality", "ingestion_date")

# 3. 
drivers_silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_drivers")

display(drivers_silver_df)

In [0]:
# List files in the drivers directory to find the correct JSON file
files = dbutils.fs.ls("/Volumes/workspace/raw_data/drivers/")
display(files)

In [0]:
from pyspark.sql.functions import col

# 1. Load your two Silver tables (Use the table names we created!)
results_df = spark.read.table("workspace.default.silver_results")
drivers_df = spark.read.table("workspace.default.silver_drivers")

# 2. Join them together
# We want to match rows where the driver_id is the same in both tables
joined_df = results_df.join(drivers_df, results_df.driver_id == drivers_df.driver_id, "inner")

# 3. Filter for the Winner of Race 1
# We need race_id to be 1 AND position_order to be 1
winner_df = joined_df.filter((col("race_id") == 1) & (col("position_order") == 1))

# 4. Show the result
# Select only the 'name' and 'nationality' columns to make it clean
display(winner_df.select("name", "nationality"))

In [0]:
from pyspark.sql.functions import col

# 1. Load Tables
results_df = spark.read.table("workspace.default.silver_results")
drivers_df = spark.read.table("workspace.default.silver_drivers")

# 2. Join (Connect them using driver_id)
joined_df = results_df.join(drivers_df, results_df.driver_id == drivers_df.driver_id, "inner")

# 3. Filter (Find Lewis in Race 1)
# Note: Strings need quotes like "Lewis Hamilton"
lewis_df = joined_df.filter((col("name") == "Lewis Hamiltondr") & (col("race_id") == 1))

# 4. Display the result
display(lewis_df.select("name", "position_text", "points"))

In [0]:
from pyspark.sql.functions import col

# 1. Load Tables
results_df = spark.read.table("workspace.default.silver_results")
drivers_df = spark.read.table("workspace.default.silver_drivers")

# 2. Join (Standard Join)
joined_df = results_df.join(drivers_df, results_df.driver_id == drivers_df.driver_id, "inner")

# 3. Filter for 2nd Place in Race 1
# FILL IN THE BLANKS BELOW:
runner_up_df = joined_df.filter((col("race_id") == 1) & (col("position_order") == 2))

# 4. Show the result
display(runner_up_df.select("name", "nationality", "time"))